# Sentinel-3 SLSTR Land Surface Temperature

**Date modified:** 10 October 2026

## Product overview

### Background

Digital Earth Africa (DE Africa) provides free and open access to [Sentinel-3 SLSTR Level-2 Land Surface Temperature](https://explorer.digitalearth.africa/products/s3_slstr_l2_lst) data over Africa. These products are generated through the European Space Agency (ESA) land processing branch of the Copernicus Sentinel-3 mission.

The [Sentinel-3 mission](https://sentiwiki.copernicus.eu/web/s3-mission), jointly operated by ESA and the European Organisation for the Exploitation of Meteorological Satellites (EUMETSAT), monitors the Earth’s oceans and land surfaces. Sentinel-3A and Sentinel-3B were launched in February 2016 and April 2018, respectively. Both satellites carry the Sea and Land Surface Temperature Radiometer (SLSTR), which acquires observations for monitoring surface temperature and environmental change.

The SLSTR Level-2 Land Surface Temperature product provides temperature estimates at approximately 1 km spatial resolution, with an approximately daily revisit frequency when observations from Sentinel-3A and Sentinel-3B are combined. Revisit frequency does not guarantee valid daily observations, as cloud cover affects data availability.

Land surface temperature is retrieved using a split-window algorithm applied to thermal infrared observations near 11 and 12 µm. The processing accounts for atmospheric effects and surface emissivity through information on atmospheric water vapour, viewing geometry, land cover and vegetation. The product also includes uncertainty estimates and quality flags to support filtering and interpretation. Further details are available in the [SLSTR processing documentation](https://sentiwiki.copernicus.eu/web/slstr-processing).

The DE Africa Sentinel-3 SLSTR Land Surface Temperature dataset includes:

* Land Surface Temperature (LST), expressed in Kelvin
* Land Surface Temperature uncertainty, expressed in Kelvin
* Normalised Difference Vegetation Index (NDVI)
* Confidence flags, including information on clouds, land, water and day/night conditions.

This document provides technical specifications for the Sentinel-3 SLSTR Level-2 Land Surface Temperature product, available in the DE Africa datacube as `s3_slstr_l2_lst`.

### Specifications

#### Spatial and temporal coverage

Relevant coverage and metadata for each of the datasets can be viewed on DE Africa [Metadata Explorer](https://explorer.digitalearth.africa/products):

- [s3_slstr_l2_lst:](https://explorer.digitalearth.africa/products/s3_slstr_l2_lst)

A [Jupyter Notebook](https://github.com/digitalearthafrica/deafrica-sandbox-notebooks/blob/main/Datasets/Sentinel_3_Land_Surface_Temperature.ipynb) which demonstrates loading and using Landsat surface temperature products in the Sandbox is also available.

**Table 1:  Sentinel-3 SLSTR Land Surface Temperature product specifications**

| Specification                   |                     Sentinel-3 SLSTR Level-2 Land Surface Temperature                     |
| :------------------------------ | :---------------------------------------------------------------------------------------: |
| **Cell size (metres)**          |                      Approximately 1,000 × 1,000 (native resolution)                      |
| **Coordinate Reference System** |                              EPSG:4326 (WGS 84) in DE Africa                              |
| **Temporal resolution**         | Approximately daily with Sentinel-3A/B combined; valid observations depend on cloud cover |
| **Parent dataset**              |                        Sentinel-3A/B SLSTR Level-2 LST (`SL_2_LST`)                       |
| **Update frequency**            |           Depends on DE Africa ingestion; delivery latency requires confirmation          |
| **Access**                      |                                Free and open via DE Africa                                |
| **Product name**                |                                     `s3_slstr_l2_lst`                                     |
| **Temperature units**           |                                         Kelvin (K)                                        |


### Measurements

#### Sentinel-3 SLSTR Land Surface Temperature Measurements – Descriptions

* **LST (Land Surface Temperature)**
  Radiometric temperature of the land surface, including soil and vegetation canopies, expressed in Kelvin (K). Convert to degrees Celsius by subtracting 273.15.

* **LST_uncertainty (Land Surface Temperature Uncertainty)**
  Total standard uncertainty associated with each LST estimate, expressed in Kelvin (K).

* **NDVI (Normalised Difference Vegetation Index)**
  Dimensionless index describing vegetation greenness, used as supporting information in land surface temperature retrieval.

* **CONFIDENCE (Confidence Flags)**
  Per-pixel bitmask identifying conditions such as land, water, cloud, snow and day/night observations to support filtering and interpretation.

**Table 2: Sentinel-3 SLSTR Level-2 Land Surface Temperature Measurements**

| Measurement                                                | Data Type             | Confidence / Quality Flags |
| :--------------------------------------------------------- | :-------------------- | :------------------------: |
| **LST** (Land Surface Temperature)                         | Temperature Parameter |             No             |
| **LST_uncertainty** (Land Surface Temperature Uncertainty) | Uncertainty Parameter |             No             |
| **NDVI** (Normalised Difference Vegetation Index)          | Vegetation Parameter  |             No             |
| **CONFIDENCE** (Confidence Flags)                          | Quality / Mask Info   |             Yes            |

These are the four measurements listed for the DE Africa product `s3_slstr_l2_lst`. A separate `dataMask` is not listed; valid-data masks can be derived from missing values and the confidence flags.

#### Understanding the Confidence layer and bitmasking

The Sentinel-3 LST data is only available over land (not ocean) and is sensitive to cloud and other interference. In the `xarray` object `ds_S3` above, we can see several 65535 values in the LST variable which represent `nodata`. We need to effectively mask the dataset before proceeding.

The `CONFIDENCE` layer uses a bitmask to encode pixel-level flags, such as cloud presence or sun glint, into a single integer. Bitmask refers to binary digits, where the position of the digit in a series relates to a particular flag. In the `CONFIDENCE` layer, flags are marked true or false with 1s and 0s at specific locations in an integer. Looking at the table below, a 1 at bit location 1 (noting 0 indexing and reading left-to-right, this would be `-1-------...`) would indicate the pixel is over ocean, while a 0 at bit location 3 (`---0-----...`) would indicate the pixel _is not_ over land.

Each active flag contributes a value of $2^{\text{position}}$ to the total integer. For example, if a pixel was over land (position 3, value $2^3=8$), had cosmetic defects (position 8, value $2^8=256$), and was cloudy (position 14, value $2^{14}=16384$), the resulting integer would be:

$$
2^3 + 2^8 + 2^{14} = 16648
$$

Looking at the `CONFIDENCE` band in the `ds_S3` object above, there are a few 16648 values. We have to work backwards from this value to find the unique combination of bit positions with raised flags. Fortunately, computers do this easily and we can draw on the datacube `masking` functions to assist.

The table below shows the bits in the Sentinel-3 LST product.

**Table 3: Sentinel-3 SLSTR Confidence Flag Definitions**

| Flag                 | Bit Position | Description                              |
| :------------------- | :----------: | :--------------------------------------- |
| **coastline**        |       0      | Pixel identified as coastline            |
| **ocean**            |       1      | Pixel identified as ocean                |
| **tidal**            |       2      | Pixel identified as a tidal area         |
| **land**             |       3      | Pixel identified as land                 |
| **inland_water**     |       4      | Pixel identified as inland water         |
| **unfilled**         |       5      | Pixel contains no filled measurement     |
| **cosmetic**         |       8      | Pixel affected by a cosmetic defect      |
| **duplicate**        |       9      | Pixel identified as a duplicate          |
| **day**              |      10      | Observation acquired during daytime      |
| **twilight**         |      11      | Observation acquired during twilight     |
| **sun_glint**        |      12      | Pixel flagged for sun glint              |
| **snow**             |      13      | Pixel flagged for snow                   |
| **summary_cloud**    |      14      | Summary flag indicating cloud            |
| **summary_pointing** |      15      | Summary flag indicating a pointing issue |

For land surface temperature analysis, use these flags to select land pixels and exclude cloud, unfilled pixels and other relevant quality issues. The **day** flag can support separate daytime and nighttime analyses; twilight observations should also be considered when defining these groups.

### Geographic Extent

Specific temporal and geographic extents for the  product can be explored interactively using the Digital Earth Africa [Metadata Explorer](https://explorer.digitalearth.africa/s3_slstr_l2_lst). 

<img src="../_static/data_specs/Sentinel-3_OLCI_L2_LAND_specs/sen3_extent.png" alt="Sentinel-3_LST data extent" width="500" align="left"/>

**Figure 1: Sentinel-3 LST L2 geographic extent**

### Processing

Sentinel-3 SLSTR Level-2 Land Surface Temperature (LST) is derived from calibrated Level-1 thermal infrared observations using a **split-window algorithm**, combining measurements near 11 and 12 µm. The retrieval accounts for atmospheric water vapour, satellite viewing angle and surface emissivity, using land cover and vegetation information. Further details are available in the [SLSTR processing documentation](https://sentiwiki.copernicus.eu/web/slstr-processing).

### Media and example images

<img src="../_static/data_specs/Sentinel-3_Land_Surface_Temperature_specs/lst.png" alt="Landsat surface temperature" width="800" align="left"/>

**Figure 2: Surface temperatures appear higher along the coast and in the low-lying areas of the delta, while cooler temperatures occur across the higher terrain in the north-western part of the area of interest. This pattern is most evident in the images from 2 and 14 April.**

<img src="../_static/data_specs/Sentinel-3_Land_Surface_Temperature_specs/lst_uncertainty.png" alt="surface temperature uncertainty " width="800" align="left"/>

**Figure 3: The uncertainty values range from 0 to 5 K. Notably, uncertainty appears higher over surface water, particularly along the Zambezi River, in the images from 2 and 5 April.**

### License

CC BY Attribution 4.0 International License

### Acknowledgements

The Sentinel-3 OLCI Level-2 SLSTR Land Surface Temperature products are produced by the European Space Agency (ESA) within the framework of the Copernicus Programme, and are freely provided via the Copernicus Open Access Hub and Copernicus Data Space Ecosystem. Data access and regional support are enabled by Digital Earth Africa.

## Data access

### Amazon Web Services S3 

The Digital Earth Africa Sentinel-3 OLCI Level-2 SLSTR Land Surface Temperature product can be accessed from the associated S3 bucket.

**Table 4: AWS data access details.**

|AWS S3 details | |
|----------|-------------|
|Bucket ARD | `arn:aws:s3:::deafrica-sentinel-3`|
|Region | `af-south-1` |
| Product name| `s3_slstr_l2_lst` |

**Table 5: AWS file path convention.**

|File path element |  Description |Example |
|----------|-------------|-----------------|
|`product name`| `s3_slstr_l2_lst`| | 
|`version`| Product version | `1-0-0` |
| `path` |  path number ranging from `157` to `206`.|`165` | 
| `row` |  row number ranging from `052` to `072`. | `052` |
| `year` | Year the data was collected |`2019` |
| `month`|  Month of the year the data was collected (with leading zeros)|`03`|
|`day`| Day of the month the data was collected (with leading zeros)|`28` |

### OGC Web Services (OWS)

This product is available through DE Africa's OWS.

**Table 6: OWS data access details.**

|OWS details | |
|----------|-------------|
|Name | `DE Africa Services` |
|Web Map Services (WMS) URL | `https://ows.digitalearth.africa/wms?version=1.3.0` |
| Web Coverage Service (WCS) URL | `https://ows.digitalearth.africa/wcs?version=2.1.0`|
| Layer name | `s3_slstr_l2_lst` |

Digital Earth Africa OWS details can be found at [https://ows.digitalearth.africa/](https://ows.digitalearth.africa/).

For instructions on how to connect to OWS, see [this tutorial](../web_services/index.ipynb).

### Open Data Cube (ODC)

The Sentinel-3 OLCI Level-2 SLSTR Land Surface Temperature collection can be accessed through the Digital Earth Africa ODC API, which is available through the [Digital Earth Africa Sandbox](https://sandbox.digitalearth.africa/hub/login).

**ODC product name:**  `s3_slstr_l2_lst`